# Caso de Estudio: Sistemas de Recomendación Tradicionales con Filtrado Colaborativo

### 1. El Rol Estratégico de los Sistemas de Recomendación Tradicionales en las Organizaciones
>1. En los procesos de transformación digital, uno de los mayores activos de una compañía es el **historial de interacciones de sus clientes** (reproducciones, compras, valoraciones y clics). Los sistemas de recomendación tradicionales, basados en **Filtrado Colaborativo `Collaborative Filtering`**, fueron los pioneros en resolver el problema de sobreabundancia de opciones (*information overload*) y cimentaron el éxito comercial de empresas como Amazon y Netflix (mediante el famoso `Netflix Prize`).

>2. Para los líderes de transformación digital, este enfoque ofrece ventajas competitivas esenciales:
>- **Descubrimiento Serendipitoso (descubrir algo inesperado) a partir de Patrones Colectivos:** El filtrado colaborativo asume que si dos usuarios coincidieron en sus preferencias pasadas, es altamente probable que coincidan en elecciones futuras, permitiendo sugerir canciones o productos sin necesidad de analizar manualmente su contenido o letra.
>- **Alineación con el *Customer Lifetime Value* (CLV):** Al anticipar qué elementos disfrutará un usuario con alta probabilidad de calificación `Rating Estimation`, las plataformas maximizan el tiempo de retención y la frecuencia de uso.
>- **Eficiencia Computacional en Producción:** Algoritmos de factorización matricial como **SVD `Singular Value Decomposition`** permiten comprimir matrices gigantescas de millones de usuarios e ítems en vectores latentes compactos de bajo costo computacional.

### 2. Fundamento Matemático y Tecnológico: Factorización Matricial con SVD

>-La biblioteca **`scikit-surprise` (Surprise)** es el estándar de referencia en Python para el desarrollo y evaluación de sistemas de recomendación colaborativos explícitos.

#### ¿Cómo funciona SVD en Recomendación?
>- Dado un conjunto de usuarios $U$ e ítems/canciones $I$, se modela la matriz de calificaciones $R \in \mathbb{R}^{|U| \times |I|}$. Como la mayoría de usuarios solo ha escuchado una pequeña fracción del catálogo musical, $R$ es una matriz extremadamente dispersa (*sparse matrix*).

>- SVD descompone la interacción entre el usuario $u$ y la canción $i$ mediante factores latentes:
$$\hat{r}_{ui} = \mu + b_u + b_i + p_u^T q_i$$
> Donde:
- $\mu$: Media global de valoración en toda la plataforma.
- $b_u$: Sesgo del usuario (usuarios sistemáticamente generosos o críticos).
- $b_i$: Sesgo del ítem (canciones universalmente aclamadas o impopulares).
- $p_u \in \mathbb{R}^k$: Vector de preferencias latentes del usuario en $k$ dimensiones.
- $q_i \in \mathbb{R}^k$: Vector de características latentes de la canción en $k$ dimensiones.

```
Flujo de Filtrado Colaborativo Tradicional Con Surprise
             
  [Matriz Usuario-Canción]          [Factorización SVD]             [Top-N Recomendación]
 ┌────────────────────────┐      ┌────────────────────────┐      ┌────────────────────────┐
 │ User ID | Song | Score │ ───► │ Aprendizaje de Factores│ ───► │ Predicción de Ratings  │
 │ (Matriz Dispersa)      │      │ Latentes (p_u, q_i)    │      │ No Observados + Filtro │
 └────────────────────────┘      └────────────────────────┘      └────────────────────────┘
```

### 3. Matriz Comparativa: Recomendador Tradicional (Surprise) versus Recomendador Semántico (RAG)

En un comité directivo de transformación digital es relevante discernir cuándo implementar cada arquitectura:

| Criterio Estratégico | Recomendador Tradicional (Surprise / SVD) | Recomendador Semántico (RAG / LLM) |
| :--- | :--- | :--- |
| **Fuente de Información** | Historial de calificaciones e interacciones numéricas `($u, i, r$)`. | Letras de canciones, biografías, metadatos y significado poético. |
| **Problema de Arranque en Frío (*Cold Start*)** | Alto: No puede recomendar canciones nuevas sin historial previo. | Bajo: Puede recomendar inmediatamente canciones nuevas analizando su letra. |
| **`Explainable Artificial Intelligence, XAI`** | Baja explicabilidad: 'Caja negra' matemática basada en factores latentes abstractos. | Alta: Genera argumentos en lenguaje natural explicando el motivo de la elección. |
| **Costo Computacional** | Muy bajo: Factorización matricial ligera en `CPU`. | Medio-Alto: Búsqueda vectorial densa e inferencia de modelos de lenguaje con `GPU`. |
| **Caso de Uso Óptimo** | Plataformas masivas con millones de clics/`ratings` registrados. | Experiencias conversacionales, descubrimiento temático y estado emocional. `NLP - review`, `embeddings - Numerical representations`, `RAG - data base`, `LLM - Recommendation`|

Instale la biblioteca [scikit-surprise](https://pypi.org/project/scikit-surprise/).

In [ ]:
# Importación de bibliotecas para el motor de recomendación colaborativo
import warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from IPython.display import display
import gradio as gr

# Importación de scikit-surprise para filtrado colaborativo 
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

warnings.filterwarnings("ignore")

print("Sistema de recomendación tradicional configurado con la biblioteca Surprise.")

## 4. Ingesta y transformación de Datos: *Music Dataset: Song Information and Lyrics*

>- Para modelar el filtrado colaborativo se requiere la tríada canónica: `(user_id, item_id, rating)`.
>- A partir del catálogo musical de canciones y su métrica de popularidad/valoración (`Popularity`), generamos un registro de interacciones corporativas representativo para calibrar la matriz de factorización. Descarga el dataset disponible en [Music Dataset: Song Information and Lyrics](https://www.kaggle.com/datasets/suraj520/music-dataset-song-information-and-lyrics).

In [ ]:
# Cargar el archivo de canciones en el repositorio o almacenamiento
data_path = "../datasets/songs.csv"

data_raw = pd.read_csv(data_path)
print(f"Catálogo musical cargado desde: {data_path}")
data_raw = pd.DataFrame(data_raw)
data_raw.head()

In [ ]:
# # Limpieza de valores nulos (Not a Number) para proteger la integridad del pipeline (user_id, item_id, rating, popularity (popularidad))
data_raw["Name"] = data_raw["Name"].fillna("Canción Desconocida")
data_raw["Artist"] = data_raw["Artist"].fillna("Artista Varios")
data_raw["Album"] = data_raw["Album"].fillna("Álbum Estándar")
data_raw["Popularity"] = data_raw["Popularity"].fillna(50).astype(float)

# Creación de un identificador único de canción (Item ID)
data_raw["song_id"] = data_raw["Name"] + " - " + data_raw["Artist"]

print(f"Catálogo consolidado: {len(data_raw):,} canciones disponibles.")
display(data_raw.head())

### 4.1 Generación de la Matriz de Interacciones Usuario-Canción ` (user_id - rating)`

>- Para entrenar el modelo colaborativo simulamos la actividad de un segmento de usuarios que escuchan y califican canciones (escala de satisfacción de 1 a 5 estrellas o rating normalizado derivado del *engagement*).

In [ ]:
np.random.seed(42)
n_usuarios = 80
n_interacciones = 1200

# Muestra de canciones del catálogo para generar registros de escucha
canciones_unicas = data_raw["song_id"].unique().tolist()
display("Canciones únicas: " + str(len(canciones_unicas)))

usuarios = [f"user_{i:03d}" for i in range(1, n_usuarios + 1)] # ID de tres digitos
interacciones = []

for _ in range(n_interacciones):
    u = np.random.choice(usuarios)
    s = np.random.choice(canciones_unicas)
    # Calificación entre 1.0 y 5.0 derivada del nivel de satisfacción
    r = np.round(np.random.choice([2.0, 3.0, 4.0, 5.0], p=[0.1, 0.25, 0.4, 0.25]), 1)
    interacciones.append({"user_id": u, "song_id": s, "rating": r})

df_ratings = pd.DataFrame(interacciones).drop_duplicates(subset=["user_id", "song_id"])
print(f"Matriz de interacción: {len(df_ratings):,} calificaciones registradas.")
display(df_ratings.head(6))

## 5. Entrenamiento del Modelo de Factorización Matricial con SVD

1. **`Reader`:** Define la escala numérica admisible para las valoraciones ($1.0$ a $5.0$).
2. **`Dataset.load_from_df`:** Estructura la matriz dispersa optimizada en memoria.
3. **Partición de Validación (*Train/Test Split*):** 80% entrenamiento y 20% prueba para evaluar objetivamente la capacidad de predicción de calificaciones no observadas.
4. **SVD con Hiperparámetros:** Calibramos $n\_factors=50$ (dimensiones latentes de gusto musical) y optimizador por descenso de gradiente estocástico.

In [ ]:
# 1. Definición del lector de Surprise con la escala de ratings
lector = Reader(rating_scale=(1.0, 5.0))

# 2. Carga estructurada de la matriz colaborativa
dataset_surprise = Dataset.load_from_df(df_ratings[["user_id", "song_id", "rating"]], lector)

# 3. Partición 80/20 para entrenamiento y auditoría
trainset, testset = train_test_split(dataset_surprise, test_size=0.20, random_state=42)

# 4. Instanciación y ajuste del algoritmo SVD
modelo_svd = SVD(n_factors=50, n_epochs=25, lr_all=0.005, reg_all=0.02, random_state=42)
modelo_svd.fit(trainset)

# 5. Evaluación cuantitativa del margen de error
predicciones_test = modelo_svd.test(testset)
rmse_eval = accuracy.rmse(predicciones_test, verbose=False)
mae_eval = accuracy.mae(predicciones_test, verbose=False)

print("Reporte de EEvaluación del modelo Singular Value Decomposition (SVD)")
print(f"Root Mean Square Error (RMSE): {rmse_eval:.4f}")
print(f"Error Absoluto Medio (MAE):    {mae_eval:.4f}")

## 6. Algoritmo Top-N de Recomendación Personalizada

>- Para un usuario determinado (o un usuario activo en sesión), el motor de recomendación calcula la **calificación esperada ($\hat{r}_{ui}$)** sobre todas las canciones del catálogo que aún no ha escuchado, ordenándolas en forma descendente para entregar la **Lista Top-N** de mayor afinidad matemática.

In [ ]:
def obtener_top_recomendaciones_svd(user_id, canciones_ya_escuchadas=None, top_n=5, album_filtro=None):
    """
    Genera las mejores recomendaciones para un usuario estimando el rating
    de todas las canciones candidatas mediante SVD.
    """
    if canciones_ya_escuchadas is None:
        canciones_ya_escuchadas = []
    
    # Filtrado opcional por álbum o categoría de catálogo
    if album_filtro and album_filtro != "Todos":
        catalogo_candidato = data_raw[data_raw["Album"] == album_filtro]["song_id"].unique().tolist()
    else:
        catalogo_candidato = data_raw["song_id"].unique().tolist()
    
    predicciones_scores = defaultdict(float)
    for cancion in catalogo_candidato:
        if cancion not in canciones_ya_escuchadas:
            # Estimación del rating esperado por el modelo colaborativo
            estimacion = modelo_svd.predict(uid=user_id, iid=cancion).est
            predicciones_scores[cancion] = estimacion
    
    # Ordenamiento por la mayor calificación predicha
    ranking = sorted(predicciones_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return ranking

In [ ]:
# Prueba de inferencia para un usuario de la plataforma
usuario_test = "user_001"
top_sugerencias = obtener_top_recomendaciones_svd(usuario_test, top_n=5)

print(f"Top 5 Canciones recomendadas para {usuario_test}:")
for pos, (cancion, score) in enumerate(top_sugerencias, start=1):
    print(f"{pos}. {cancion} | Rating Estimado: {score:.2f} / 5.0")

## 7. Dashboard de Métricas y Diagnóstico para la Gobernanza Analítica

En la gestión estratégica de la transformación digital, los modelos colaborativos deben monitorearse bajo un marco integral de cuatro dimensiones:
1. **Distribución de Residuos del Error:** Contrastar la certeza en la estimación de ratings.
2. **Sensibilidad del Parámetro Latente ($k$):** Evaluar cómo varía el RMSE al modificar el número de factores latentes (`n_factors`).
3. **Curva de Concentración de Recomendaciones:** Diagnosticar si el algoritmo sufre de sesgo de popularidad o si activa el catálogo completo.
4. **Benchmarking de Velocidad e Inferencia:** Evaluar la latencia por recomendación (< 1 ms en SVD) para validar despliegue masivo en tiempo real.

## 8. Despliegue de la Experiencia Interactiva con Gradio

Desplegamos una interfaz gráfica interactiva que simula la consola de personalización en la plataforma corporativa. El usuario selecciona su perfil, excluye pistas ya conocidas y recibe de inmediato su **Top 5 de Recomendaciones Colaborativas** calculadas con SVD.

In [ ]:
# Lista única de álbumes para filtro contextual
albumes_lista = ["Todos"] + sorted(data_raw["Album"].unique().tolist())
usuarios_disponibles = sorted(df_ratings["user_id"].unique().tolist())[:30]

def interfaz_recomendar(usuario_sel, album_sel, canciones_excluir):
    sugerencias = obtener_top_recomendaciones_svd(
        user_id=usuario_sel,
        canciones_ya_escuchadas=canciones_excluir,
        top_n=5,
        album_filtro=album_sel
    )
    
    lineas = [
        f"**{i}. {cancion}** — Rating Estimado: `{score:.2f} / 5.0`"
        for i, (cancion, score) in enumerate(sugerencias, start=1)
    ]
    
    resultado = (
        f"### Lista Top 5 Personalizada para `{usuario_sel}`:\n" +
        "\n".join(lineas) + "\n\n" +
        "> **Gobernanza de Machine Learning:** Este resultado fue generado mediante factorización matricial SVD " +
        "descomponiendo patrones colectivos de interacción sin requerir análisis de texto."
    )
    return resultado

with gr.Blocks(theme=gr.themes.Soft(), title="Recomendador Tradicional SVD") as demo_surprise:
    gr.Markdown("# Sistema de Recomendación Tradicional con Filtrado Colaborativo (SVD)")
    gr.Markdown(
        "**Maestría en Liderazgo y Gestión Estratégica de la Transformación Digital**\n\n"
        "Este laboratorio ilustra cómo las plataformas de comercio electrónico y streaming utilizan "
        "factorización matricial rápida para predecir afinidades y personalizar la experiencia del usuario."
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            dropdown_user = gr.Dropdown(choices=usuarios_disponibles, value=usuarios_disponibles[0], label="Seleccione Usuario Registrado")
            dropdown_album = gr.Dropdown(choices=albumes_lista, value="Todos", label="Filtrar por Álbum / Categoría")
            check_excluir = gr.CheckboxGroup(choices=canciones_unicas[:10], label="Canciones que el usuario ya conoce (excluir de la recomendación)")
            btn_calc = gr.Button("Obtener Recomendaciones Colaborativas", variant="primary")
            
        with gr.Column(scale=2):
            output_box = gr.Markdown(label="Resultado de Recomendación")
            
    btn_calc.click(
        fn=interfaz_recomendar,
        inputs=[dropdown_user, dropdown_album, check_excluir],
        outputs=[output_box]
    )

# Lanzamiento de la interfaz web en modo embebido
demo_surprise.launch(inline=True)

## 9. Directrices de Liderazgo y Transformación Digital para la Alta Gerencia

| Dimensión Estratégica | Reto Organizacional | Solución con Filtrado Colaborativo |
| :--- | :--- | :--- |
| **Escalabilidad y Costo Cloud** | Plataformas con millones de usuarios donde el costo de invocar modelos LLM por cada clic es prohibitivo. | Desplegar SVD en arquitecturas desacopladas: el costo computacional por recomendación es inferior a 1 milisegundo en CPUs estándar. |
| **Efecto Red (*Network Effects*)** | Las plataformas crecen cuando el uso de un cliente enriquece la experiencia de los demás. | El filtrado colaborativo se auto-alimenta: a mayor volumen de transacciones, más precisos son los factores latentes $p_u$ y $q_i$. |
| **Arquitecturas Híbridas (El Estado del Arte)** | El filtrado colaborativo falla con canciones nuevas (*Cold Start*) y el RAG tiene alta latencia. | Implementar un **Pipeline Híbrido en 2 Etapas**:  <br>1. *Candidatos Masivos (SVD):* Filtra de 100,000 ítems a los 50 más probables.  <br>2. *Reranking Semántico (RAG):* Selecciona los mejores 5 y genera la justificación explicable para el usuario. |